# Pompano Beach Data Cleaning Pipeline
## City-Specific Data Normalization

This notebook handles **Pompano Beach specific** data cleaning and normalization:
- Loads raw JSON files from `results_folder/pompano/raw_json/`
- Extracts tables from JSON chunks
- Applies Pompano-specific field mappings
- Outputs clean, normalized data ready for financial analysis

**Input**: Raw JSON files from extraction pipeline  
**Output**: Clean, normalized CSV ready for joining with other cities

## Environment Setup

In [ ]:
import pandas as pd
import json
from pathlib import Path
from io import StringIO
import sys

# Add parent directory to path for imports
sys.path.append(str(Path.cwd().parent))

# Import functions directly to avoid relative import issues
try:
    from normalizers import normalize_pompano
    print("Successfully imported normalize_pompano")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Let's import the functions we need directly...")
    
    # If normalizers import fails, we'll define a simple version here
    def normalize_pompano(df, city, source_file=None):
        """Simple Pompano normalizer"""
        rename_map = {
            "Formatted Case Number": "violation_id_raw",
            "Address": "address_raw",
            "Violation Code": "violation_code_raw",
            "Violation Description": "violation_description_raw",
            "Case Disposition": "case_disposition_raw",
            "Case Status Description": "case_status_raw",
            "Case Established Date": "opened_date_raw",
            "Days Active": "days_active_raw",
            "Last Action": "last_action_raw",
            "Result Date": "result_date_raw",
            "Next Action": "next_action_raw",
            "Due Date": "due_date_raw"
        }
        df2 = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}).copy()
        df2["city"] = city
        df2["source_file"] = source_file
        return df2

# Path masking function
def mask_path(p: str | Path) -> str:
    p = Path(p)
    parts = p.parts
    return str(Path(*(["..."] + list(parts[-3:]))))

# Project paths
if "__file__" in globals():
    ROOT = Path(__file__).resolve().parents[2]
else:
    cwd = Path.cwd()
    if cwd.name == "cleaning":
        ROOT = cwd.parents[1]
    elif cwd.name == "src":
        ROOT = cwd.parent
    else:
        ROOT = cwd

RESULTS_DIR = ROOT / "results_folder"
POMPANO_DIR = RESULTS_DIR / "pompano"
CLEAN_DIR = ROOT / "clean_data"
CLEAN_DIR.mkdir(exist_ok=True)

print("ROOT:", mask_path(ROOT))
print("POMPANO_DIR:", mask_path(POMPANO_DIR))
print("CLEAN_DIR:", mask_path(CLEAN_DIR))

# Find JSON files
json_dir = POMPANO_DIR / "raw_json"
if json_dir.exists():
    json_files = list(json_dir.glob("*.json"))
    print(f"Found {len(json_files)} JSON files")
    for f in json_files:
        print(f"  - {f.name}")
else:
    print(f"❌ JSON directory not found: {mask_path(json_dir)}")

❌ Import error: attempted relative import with no known parent package
Let's import the functions we need directly...
ROOT: ...\Edilma Projects\LandingAI-Hack\coderisk-sf
POMPANO_DIR: ...\coderisk-sf\results_folder\pompano
CLEAN_DIR: ...\LandingAI-Hack\coderisk-sf\clean_data
 Found 1 JSON files
  - pompanoViolations_20240124-30p.json


## Extract Tables from JSON Chunks

In [3]:
def extract_pompano_tables(json_file_path):
    """
    Extract tables from Pompano JSON chunks with proper header handling.
    """
    print(f"Processing Pompano JSON: {Path(json_file_path).name}")
    
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    all_tables = []
    
    if 'chunks' in data:
        for i, chunk in enumerate(data['chunks']):
            if chunk.get('type') == 'table' and 'markdown' in chunk:
                print(f"  Processing table chunk {i+1}")
                
                # Parse HTML table with proper header handling
                df = parse_html_table(chunk['markdown'])
                if df is not None and not df.empty:
                    # Add metadata with masked path
                    df['chunk_id'] = i
                    df['source_file'] = mask_path(json_file_path)
                    all_tables.append(df)
                    print(f"    Extracted {len(df)} rows")
    
    if all_tables:
        combined_df = pd.concat(all_tables, ignore_index=True)
        print(f"Total extracted: {len(combined_df)} rows")
        return combined_df
    else:
        print("❌ No table chunks found")
        return pd.DataFrame()

def parse_html_table(html_content):
    """Parse HTML table with proper header detection for Pompano data"""
    try:
        if '<table' in html_content.lower():
            # Parse table without assuming header location
            tables = pd.read_html(StringIO(html_content), header=None)
            if tables:
                df = tables[0]
                
                # Use first row as column names and remove it from data
                if len(df) > 1:
                    headers = df.iloc[0].fillna('').astype(str).tolist()
                    data_df = df.iloc[1:].reset_index(drop=True)
                    data_df.columns = headers[:len(data_df.columns)]
                    return data_df
                else:
                    return df
    except Exception as e:
        print(f"    ❌ Parse error: {e}")
    return None

## Process All Pompano Files

In [4]:
# Process all Pompano JSON files
print("Processing ALL Pompano files...")

all_data = []

for i, json_file in enumerate(json_files, 1):
    print(f"\n[{i}/{len(json_files)}] Processing: {json_file.name}")
    df = extract_pompano_tables(json_file)
    if not df.empty:
        all_data.append(df)
        print(f"  Got {len(df)} rows")
    else:
        print(f"  ❌ No data from this file")

if all_data:
    # Combine all data
    pompano_raw = pd.concat(all_data, ignore_index=True)
    
    print(f"\nRAW POMPANO DATA:")
    print(f"Shape: {pompano_raw.shape}")
    print(f"Columns: {list(pompano_raw.columns)}")
    print(f"\nSample data:")
    print(pompano_raw.head(3))
    
else:
    print("❌ No data extracted from any files")
    pompano_raw = pd.DataFrame()

Processing ALL Pompano files...

[1/1] Processing: pompanoViolations_20240124-30p.json
Processing Pompano JSON: pompanoViolations_20240124-30p.json
  Processing table chunk 5
    Extracted 12 rows
  Processing table chunk 13
    Extracted 14 rows
  Processing table chunk 21
    Extracted 13 rows
  Processing table chunk 29
    Extracted 13 rows
  Processing table chunk 37
    Extracted 14 rows
  Processing table chunk 45
    Extracted 15 rows
  Processing table chunk 53
    Extracted 14 rows
  Processing table chunk 61
    Extracted 13 rows
  Processing table chunk 69
    Extracted 13 rows
  Processing table chunk 77
    Extracted 13 rows
  Processing table chunk 85
    Extracted 12 rows
  Processing table chunk 93
    Extracted 13 rows
  Processing table chunk 101
    Extracted 13 rows
  Processing table chunk 109
    Extracted 13 rows
  Processing table chunk 117
    Extracted 13 rows
  Processing table chunk 125
    Extracted 12 rows
  Processing table chunk 133
    Extracted 12 row

## Data Cleaning

In [ ]:
if not pompano_raw.empty:
    print("Cleaning Pompano data...")
    
    pompano_clean = pompano_raw.copy()
    
    print(f"Before cleaning: {len(pompano_clean)} rows")
    
    # Remove any obvious header repeats
    if "Formatted Case Number" in pompano_clean.columns:
        header_mask = pompano_clean["Formatted Case Number"].astype(str).str.strip() == "Formatted Case Number"
        rows_before = len(pompano_clean)
        pompano_clean = pompano_clean[~header_mask]
        print(f"Removed {rows_before - len(pompano_clean)} header repeat rows")
    
    # Remove empty rows
    rows_before = len(pompano_clean)
    pompano_clean = pompano_clean.dropna(how='all')
    print(f"Removed {rows_before - len(pompano_clean)} completely empty rows")
    
    print(f"After cleaning: {len(pompano_clean)} rows")
    
    # Show data overview
    print(f"\nColumn overview:")
    for col in pompano_clean.columns:
        non_null = pompano_clean[col].notna().sum()
        print(f"  {col}: {non_null} non-null values")
    
    print(f"\nSample cleaned data:")
    print(pompano_clean.head(3))
    
else:
    print("❌ No raw data to clean")
    pompano_clean = pd.DataFrame()

 Cleaning Pompano data...
Before cleaning: 389 rows
Removed 0 header repeat rows
Removed 0 completely empty rows
After cleaning: 389 rows

 Column overview:
  Formatted Case Number: 370 non-null values
  Violation Code: 372 non-null values
  Violation Description: 378 non-null values
  Case Disposition: 370 non-null values
  Address: 378 non-null values
  Case Status Description: 370 non-null values
  Case Established Date: 370 non-null values
  Days Active: 370 non-null values
  Last Action: 348 non-null values
  Result Date: 333 non-null values
  Next Action: 374 non-null values
  Due Date: 370 non-null values
  chunk_id: 389 non-null values
  source_file: 389 non-null values

Sample cleaned data:
  Formatted Case Number        Violation Code        Violation Description  \
0           23 09003645        CO 96.26(D)(4)    NUISANCE; STATE OF REPAIR   
1           23 09003645        CO 96.26(D)(5)  NUISANCE; BARRIER CONDITION   
2           23 09003597  CO 155.5203(B)(6)(c)    LANDSCAP

## Apply Normalization

In [ ]:
if not pompano_clean.empty:
    print("Applying Pompano normalization...")
    
    # Apply the normalize_pompano function from normalizers.py
    pompano_normalized = normalize_pompano(
        pompano_clean, 
        city="Pompano Beach", 
        source_file="pompanoViolations_20240124-30p.json"
    )
    
    print(f"Normalization complete!")
    print(f"Shape: {pompano_normalized.shape}")
    print(f"Columns: {list(pompano_normalized.columns)}")
    
    # Show sample normalized data
    print(f"\nSample normalized data:")
    key_cols = ['violation_id_raw', 'address_raw', 'case_status_raw', 'opened_date_raw']
    display_cols = [col for col in key_cols if col in pompano_normalized.columns]
    print(pompano_normalized[display_cols].head(5))
    
    # Show unique case count
    if 'violation_id_raw' in pompano_normalized.columns:
        unique_cases = pompano_normalized['violation_id_raw'].nunique()
        print(f"\nFound {unique_cases} unique cases")
    
else:
    print("❌ No clean data to normalize")
    pompano_normalized = pd.DataFrame()

 Applying Pompano normalization...
Normalization complete!
Shape: (389, 15)
Columns: ['violation_id_raw', 'violation_code_raw', 'violation_description_raw', 'case_disposition_raw', 'address_raw', 'case_status_raw', 'opened_date_raw', 'days_active_raw', 'last_action_raw', 'result_date_raw', 'next_action_raw', 'due_date_raw', 'chunk_id', 'source_file', 'city']

 Sample normalized data:
  violation_id_raw        address_raw case_status_raw opened_date_raw
0      23 09003645  598 E ATLANTIC BL          ACTIVE      11/27/2023
1      23 09003645  598 E ATLANTIC BL          ACTIVE      11/27/2023
2      23 09003597  901 E ATLANTIC BL          ACTIVE      11/22/2023
3      23 09003597  901 E ATLANTIC BL          ACTIVE      11/22/2023
4      23 09003597  901 E ATLANTIC BL          ACTIVE      11/22/2023

 Found 120 unique cases


## Save Results

In [ ]:
if not pompano_normalized.empty:
    # Save cleaned data
    output_file = CLEAN_DIR / "pompano_beach_clean.csv"
    pompano_normalized.to_csv(output_file, index=False)
    
    print(f"Saved to: {mask_path(output_file)}")
    
    # Final stats
    print(f"\nPOMPANO BEACH RESULTS:")
    print(f"  - Total rows: {len(pompano_normalized)}")
    if 'violation_id_raw' in pompano_normalized.columns:
        print(f"  - Unique cases: {pompano_normalized['violation_id_raw'].nunique()}")
    if 'case_status_raw' in pompano_normalized.columns:
        print(f"  - Status breakdown:")
        status_counts = pompano_normalized['case_status_raw'].value_counts()
        for status, count in status_counts.head(5).items():
            print(f"    {status}: {count}")
    
    # Date range if available
    if 'opened_date_raw' in pompano_normalized.columns:
        date_col = pd.to_datetime(pompano_normalized['opened_date_raw'], errors='coerce')
        valid_dates = date_col.dropna()
        if not valid_dates.empty:
            print(f"  - Date range: {valid_dates.min().date()} to {valid_dates.max().date()}")
    
    print(f"\nSUCCESS! Pompano Beach data ready for financial analysis!")
    
else:
    print("❌ No data to save")

💾 Saved to: ...\coderisk-sf\clean_data\pompano_beach_clean.csv

🎯 POMPANO BEACH RESULTS:
  - Total rows: 389
  - Unique cases: 120
  - Status breakdown:
    ACTIVE: 370
  - Date range: 2023-04-25 to 2024-01-18

✅ SUCCESS! Pompano Beach data ready for financial analysis!
